# Inventory: Raw -> Bronze

Land the raw JSON extract as-is, only standardizing column names.

In [1]:
%run ../00_config.ipynb
%run ../00_utils.ipynb

/usr/local/lib/python3.12/site-packages/nbformat/validator.py:434: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  _validate(nbdict, ref, version, version_minor, relax_add_props)


[08/23/26 15:05:52] INFO     Using                                                                  ]8;id=9194620;file:///usr/local/lib/python3.12/site-packages/kedro/framework/project/__init__.py\__init__.py]8;;\:]8;id=9194621;file:///usr/local/lib/python3.12/site-packages/kedro/framework/project/__init__.py#302\302]8;;\
                             '/usr/local/lib/python3.12/site-packages/kedro/framework/project/rich_                
                             logging.yml' as logging configuration.                                                

[08/23/26 15:05:52] WARNING  /usr/local/lib/python3.12/site-packages/kedro/framework/context/contex ]8;id=9194628;file:///usr/local/lib/python3.12/warnings.py\warnings.py]8;;\:]8;id=9194629;file:///usr/local/lib/python3.12/warnings.py#112\112]8;;\
                             t.py:221: UserWarning: Parameters not found in your Kedro project                     
                             config.                                                                               
                             No files of YAML or JSON format found in /app/conf/base or                            
                             /app/conf/local matching the glob pattern(s): ['parameters*',                         
                             'parameters*/**', '**/parameters*']                                                   
                               warn(f"Parameters not found in your Kedro project config.\n{exc!s}")                
                                                                                                                   

                    INFO     No typed parameter requirements found, returning original   ]8;id=9194636;file:///usr/local/lib/python3.12/site-packages/kedro/validation/parameter_validator.py\parameter_validator.py]8;;\:]8;id=9194637;file:///usr/local/lib/python3.12/site-packages/kedro/validation/parameter_validator.py#124\124]8;;\
                             parameters                                                                            

Kedro context loaded from /app
Catalog datasets: ['raw_employees', 'bronze_employees', 'silver_employees', 'gold_employees', 'raw_sales', 'bronze_sales', 'silver_sales', 'gold_sales', 'raw_inventory', 'bronze_inventory', 'silver_inventory', 'gold_inventory', 'parameters']


                    WARNING  /usr/local/lib/python3.12/site-packages/nbformat/validator.py:434:     ]8;id=9194642;file:///usr/local/lib/python3.12/warnings.py\warnings.py]8;;\:]8;id=9194643;file:///usr/local/lib/python3.12/warnings.py#112\112]8;;\
                             MissingIDFieldWarning: Cell is missing an id field, this will become a                
                             hard error in future nbformat versions. You may want to use                           
                             `normalize()` on your notebooks before validations (available since                   
                             nbformat 5.1.4). Previous versions of nbformat are fixing this issue                  
                             transparently, and will stop doing so in the future.                                  
                               _validate(nbdict, ref, version, version_minor, relax_add_props)                     
                                                                                                                   

In [2]:
raw_inventory = catalog.load("raw_inventory")
raw_inventory

                    INFO     Loading data from raw_inventory (JSONDataset)...                  ]8;id=9194650;file:///usr/local/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=9194651;file:///usr/local/lib/python3.12/site-packages/kedro/io/data_catalog.py#1050\1050]8;;\

,item_id,item_name,category,stock_count,warehouse,last_updated,notes
0,201.0,Steel Bolt,hardware,500,WH-1,2023-01-10,restocked
1,202.0,Copper Wire,ELECTRICAL,150,WH-2,2023-02-15,
2,203.0,Rubber Gasket,hardware,300,WH-1,2023-03-01,low stock
3,NaN,Ghost Item,unknown,0,WH-0,2022-01-01,"bad data, should be dropped"
4,204.0,LED Bulb,electrical,1000,WH-3,2023-04-20,


In [3]:
bronze_inventory = standardize_columns(raw_inventory)
bronze_inventory

,item_id,item_name,category,stock_count,warehouse,last_updated,notes
0,201.0,Steel Bolt,hardware,500,WH-1,2023-01-10,restocked
1,202.0,Copper Wire,ELECTRICAL,150,WH-2,2023-02-15,
2,203.0,Rubber Gasket,hardware,300,WH-1,2023-03-01,low stock
3,NaN,Ghost Item,unknown,0,WH-0,2022-01-01,"bad data, should be dropped"
4,204.0,LED Bulb,electrical,1000,WH-3,2023-04-20,


In [4]:
catalog.save("bronze_inventory", bronze_inventory)

                    INFO     Saving data to bronze_inventory (CSVDataset)...                   ]8;id=9194657;file:///usr/local/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=9194658;file:///usr/local/lib/python3.12/site-packages/kedro/io/data_catalog.py#1006\1006]8;;\

## PySpark alternative (reference only)

PySpark isn't installed in this image. Left commented out to show how this
stage would read/write with Spark instead of pandas.

In [5]:
# raw_inventory_spark = spark.read.option("multiline", "true").json(
#     str(PROJECT_ROOT / "data/01_raw/inventory.json")
# )
#
# bronze_inventory_spark = standardize_columns_spark(raw_inventory_spark)
#
# bronze_inventory_spark.write.mode("overwrite").option("header", "true").csv(
#     str(PROJECT_ROOT / "data/02_bronze/inventory.csv")
# )